In [1]:
!pip install adjustText==1.3.0


[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [2]:
#!/usr/bin/env python3
"""
6_INTERNET_USAGE_REGRESSION.py

Analyzes the relationship between prediction error (MAE) and internet usage.

Two types of regression:
1. Aggregated: MAE across all stages vs internet usage
2. By Stage: MAE at each stage vs internet usage
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, spearmanr
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Try to import adjustText for better label positioning
try:
    from adjustText import adjust_text
    HAS_ADJUST_TEXT = True
except ImportError:
    HAS_ADJUST_TEXT = False
    print("⚠️  adjustText not installed - country labels may overlap")
    print("   Install with: pip install adjustText")

# Set style
sns.set_style("whitegrid")
# Set default figure size - will be overridden by individual figures
import matplotlib
matplotlib.use('Agg')

print("="*80)
print("INTERNET USAGE vs PREDICTION ERROR ANALYSIS")
print("="*80)

# ================================================================
# 1. Load Data
# ================================================================

print("\n" + "="*80)
print("1. LOADING DATA")
print("="*80)

# Load predictions
df = pd.read_csv("predictions_all_stages_long.csv")
print(f"✓ Loaded predictions: {len(df)} rows")

# Load ground truth for calculating MAE
gt_df = pd.read_csv("data_final.csv")
gt_df['ground_truth_pi'] = gt_df['mean_other_willingness'] * 100
df = df.merge(gt_df[['countrynew', 'ground_truth_pi']], on='countrynew', how='left')
print(f"✓ Merged ground truth")

# Load internet usage
internet_df = pd.read_csv("matched_internet_usage.csv")
# Rename to match
internet_df = internet_df.rename(columns={'Country Name': 'countrynew'})
print(f"✓ Loaded internet usage: {len(internet_df)} countries")

# Check for missing internet usage
missing_internet = internet_df[internet_df['internet_usage'].isna()]['countrynew'].tolist()
if missing_internet:
    print(f"\n⚠️  Warning: {len(missing_internet)} countries missing internet usage:")
    print(f"   {missing_internet}")

# Merge internet usage
df = df.merge(internet_df, on='countrynew', how='left')

# Remove rows with missing data
df_complete = df[df['internet_usage'].notna() & df['ground_truth_pi'].notna()].copy()
print(f"\n✓ Final dataset: {len(df_complete)} rows")
print(f"   Countries: {df_complete['countrynew'].nunique()}")
print(f"   Internet usage range: {df_complete['internet_usage'].min():.1f}% - {df_complete['internet_usage'].max():.1f}%")

# ================================================================
# 2. Calculate MAE for each model
# ================================================================

print("\n" + "="*80)
print("2. CALCULATING MAE")
print("="*80)

models = ['gpt', 'claude', 'gemini', 'llama']

for model in models:
    pred_col = f'pred_{model}'
    mae_col = f'mae_{model}'
    df_complete[mae_col] = abs(df_complete[pred_col] - df_complete['ground_truth_pi'])

print("✓ Calculated MAE for all models")

# ================================================================
# 3. Aggregate Analysis (across all stages)
# ================================================================

print("\n" + "="*80)
print("3. AGGREGATE REGRESSION (Across All Stages)")
print("="*80)

# Calculate mean MAE across all stages for each country
country_aggregate = df_complete.groupby('countrynew').agg({
    'mae_gpt': 'mean',
    'mae_claude': 'mean',
    'mae_gemini': 'mean',
    'mae_llama': 'mean',
    'internet_usage': 'first'  # Same for all stages
}).reset_index()

print(f"\n✓ Aggregated data: {len(country_aggregate)} countries")

# Run regressions for each model
aggregate_results = []

for model in models:
    mae_col = f'mae_{model}'
    
    # Remove any NaN values
    data = country_aggregate[[mae_col, 'internet_usage']].dropna()
    
    if len(data) == 0:
        print(f"\n⚠️  No data for {model.upper()}")
        continue
    
    X = data['internet_usage'].values.reshape(-1, 1)
    y = data[mae_col].values
    
    # Linear regression
    lr = LinearRegression()
    lr.fit(X, y)
    
    y_pred = lr.predict(X)
    r2 = r2_score(y, y_pred)
    
    # Pearson correlation
    pearson_r, pearson_p = pearsonr(data['internet_usage'], data[mae_col])
    
    # Spearman correlation
    spearman_r, spearman_p = spearmanr(data['internet_usage'], data[mae_col])
    
    print(f"\n{model.upper()}:")
    print(f"   N countries: {len(data)}")
    print(f"   Slope: {lr.coef_[0]:.4f} (MAE change per 1% internet usage)")
    print(f"   Intercept: {lr.intercept_:.2f}")
    print(f"   R²: {r2:.4f}")
    print(f"   Pearson r: {pearson_r:.4f}, p = {pearson_p:.4f}")
    print(f"   Spearman ρ: {spearman_r:.4f}, p = {spearman_p:.4f}")
    
    aggregate_results.append({
        'Model': model.upper(),
        'N': len(data),
        'Slope': lr.coef_[0],
        'Intercept': lr.intercept_,
        'R²': r2,
        'Pearson r': pearson_r,
        'Pearson p': pearson_p,
        'Spearman ρ': spearman_r,
        'Spearman p': spearman_p
    })

aggregate_df = pd.DataFrame(aggregate_results)

print("\n" + "="*60)
print("AGGREGATE RESULTS SUMMARY:")
print("="*60)
print(aggregate_df.to_string(index=False))

# ================================================================
# 4. By-Stage Analysis
# ================================================================

print("\n" + "="*80)
print("4. BY-STAGE REGRESSION")
print("="*80)

stage_results = []

for stage in sorted(df_complete['stage'].unique()):
    print(f"\n{'='*60}")
    print(f"Stage {stage}:")
    print('='*60)
    
    stage_data = df_complete[df_complete['stage'] == stage].copy()
    
    # Aggregate by country for this stage
    stage_country = stage_data.groupby('countrynew').agg({
        'mae_gpt': 'mean',
        'mae_claude': 'mean',
        'mae_gemini': 'mean',
        'mae_llama': 'mean',
        'internet_usage': 'first'
    }).reset_index()
    
    for model in models:
        mae_col = f'mae_{model}'
        
        # Remove any NaN values
        data = stage_country[[mae_col, 'internet_usage']].dropna()
        
        if len(data) == 0:
            continue
        
        X = data['internet_usage'].values.reshape(-1, 1)
        y = data[mae_col].values
        
        # Linear regression
        lr = LinearRegression()
        lr.fit(X, y)
        
        y_pred = lr.predict(X)
        r2 = r2_score(y, y_pred)
        
        # Pearson correlation
        pearson_r, pearson_p = pearsonr(data['internet_usage'], data[mae_col])
        
        # Spearman correlation
        spearman_r, spearman_p = spearmanr(data['internet_usage'], data[mae_col])
        
        print(f"\n   {model.upper()}:")
        print(f"      Slope: {lr.coef_[0]:.4f}, R²: {r2:.4f}, r: {pearson_r:.4f} (p={pearson_p:.4f})")
        
        stage_results.append({
            'Stage': stage,
            'Model': model.upper(),
            'N': len(data),
            'Slope': lr.coef_[0],
            'Intercept': lr.intercept_,
            'R²': r2,
            'Pearson r': pearson_r,
            'Pearson p': pearson_p,
            'Spearman ρ': spearman_r,
            'Spearman p': spearman_p
        })

stage_df = pd.DataFrame(stage_results)

# ================================================================
# 5. Country Code Mapping
# ================================================================

print("\n" + "="*80)
print("5. CREATING COUNTRY CODE MAPPING")
print("="*80)

# ISO 3-letter country codes
country_codes = {
    'Afghanistan': 'AFG', 'Albania': 'ALB', 'Algeria': 'DZA', 'Argentina': 'ARG',
    'Armenia': 'ARM', 'Australia': 'AUS', 'Austria': 'AUT', 'Bangladesh': 'BGD',
    'Belgium': 'BEL', 'Benin': 'BEN', 'Bolivia': 'BOL', 'Bosnia Herzegovina': 'BIH',
    'Botswana': 'BWA', 'Brazil': 'BRA', 'Bulgaria': 'BGR', 'Burkina Faso': 'BFA',
    'Cambodia': 'KHM', 'Cameroon': 'CMR', 'Canada': 'CAN', 'Chad': 'TCD',
    'Chile': 'CHL', 'China': 'CHN', 'Colombia': 'COL', 'Congo Brazzaville': 'COG',
    'Costa Rica': 'CRI', 'Croatia': 'HRV', 'Cyprus': 'CYP', 'Czech Republic': 'CZE',
    'Denmark': 'DNK', 'Dominican Republic': 'DOM', 'Ecuador': 'ECU', 'Egypt': 'EGY',
    'El Salvador': 'SLV', 'Estonia': 'EST', 'Ethiopia': 'ETH', 'Finland': 'FIN',
    'France': 'FRA', 'Gabon': 'GAB', 'Georgia': 'GEO', 'Germany': 'DEU',
    'Ghana': 'GHA', 'Greece': 'GRC', 'Guatemala': 'GTM', 'Guinea': 'GIN',
    'Haiti': 'HTI', 'Honduras': 'HND', 'Hong Kong': 'HKG', 'Hungary': 'HUN',
    'Iceland': 'ISL', 'India': 'IND', 'Indonesia': 'IDN', 'Iran': 'IRN',
    'Iraq': 'IRQ', 'Ireland': 'IRL', 'Israel': 'ISR', 'Italy': 'ITA',
    'Ivory Coast': 'CIV', 'Jamaica': 'JAM', 'Japan': 'JPN', 'Jordan': 'JOR',
    'Kazakhstan': 'KAZ', 'Kenya': 'KEN', 'Kosovo': 'XKX', 'Kyrgyzstan': 'KGZ',
    'Laos': 'LAO', 'Latvia': 'LVA', 'Lebanon': 'LBN', 'Liberia': 'LBR', 'Libya': 'LBY',
    'Lithuania': 'LTU', 'Luxembourg': 'LUX', 'Macedonia': 'MKD', 'Madagascar': 'MDG',
    'Malawi': 'MWI', 'Malaysia': 'MYS', 'Mali': 'MLI', 'Malta': 'MLT',
    'Mauritania': 'MRT', 'Mauritius': 'MUS', 'Mexico': 'MEX', 'Moldova': 'MDA',
    'Mongolia': 'MNG', 'Montenegro': 'MNE', 'Morocco': 'MAR', 'Mozambique': 'MOZ',
    'Myanmar': 'MMR', 'Namibia': 'NAM', 'Nepal': 'NPL', 'Netherlands': 'NLD',
    'New Zealand': 'NZL', 'Nicaragua': 'NIC', 'Niger': 'NER', 'Nigeria': 'NGA',
    'North Macedonia': 'MKD', 'Norway': 'NOR', 'Pakistan': 'PAK', 'Palestinian Territories': 'PSE',
    'Panama': 'PAN', 'Paraguay': 'PRY', 'Peru': 'PER', 'Philippines': 'PHL',
    'Poland': 'POL', 'Portugal': 'PRT', 'Romania': 'ROU', 'Russia': 'RUS', 'Rwanda': 'RWA',
    'Saudi Arabia': 'SAU', 'Senegal': 'SEN', 'Serbia': 'SRB', 'Sierra Leone': 'SLE',
    'Singapore': 'SGP', 'Slovakia': 'SVK', 'Slovenia': 'SVN', 'South Africa': 'ZAF',
    'South Korea': 'KOR', 'Spain': 'ESP', 'Sri Lanka': 'LKA', 'Sweden': 'SWE',
    'Switzerland': 'CHE', 'Taiwan': 'TWN', 'Tajikistan': 'TJK', 'Tanzania': 'TZA',
    'Thailand': 'THA', 'Togo': 'TGO', 'Tunisia': 'TUN', 'Turkey': 'TUR',
    'Turkmenistan': 'TKM', 'Uganda': 'UGA', 'Ukraine': 'UKR', 'United Arab Emirates': 'ARE',
    'United Kingdom': 'GBR', 'United States': 'USA', 'Uruguay': 'URY',
    'Uzbekistan': 'UZB', 'Venezuela': 'VEN', 'Vietnam': 'VNM', 'Yemen': 'YEM',
    'Zambia': 'ZMB', 'Zimbabwe': 'ZWE'
}

country_aggregate['country_code'] = country_aggregate['countrynew'].map(country_codes)
print(f"✓ Mapped {country_aggregate['country_code'].notna().sum()} country codes")

# ================================================================
# 6. Visualizations with Country Labels
# ================================================================

print("\n" + "="*80)
print("6. CREATING VISUALIZATIONS")
print("="*80)

from adjustText import adjust_text

# Figure 1: Aggregate scatter plots (4 models) with country labels
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
fig.suptitle('Internet Usage vs Prediction Error (Aggregated Across All Stages)', 
             fontsize=16, fontweight='bold')

for idx, model in enumerate(models):
    ax = axes[idx // 2, idx % 2]
    
    mae_col = f'mae_{model}'
    data = country_aggregate[['countrynew', 'country_code', mae_col, 'internet_usage']].dropna()
    
    # Scatter plot
    scatter = ax.scatter(data['internet_usage'], data[mae_col], 
                        alpha=0.6, s=50, c='steelblue', edgecolors='black', linewidth=0.5)
    
    # Add country labels
    texts = []
    for _, row in data.iterrows():
        if pd.notna(row['country_code']):
            texts.append(ax.text(row['internet_usage'], row[mae_col], 
                               row['country_code'], fontsize=6, alpha=0.7))
    
    # Adjust text to avoid overlaps
    if HAS_ADJUST_TEXT:
        try:
            adjust_text(texts, arrowprops=dict(arrowstyle='-', color='gray', lw=0.5, alpha=0.5),
                       ax=ax, expand_points=(1.2, 1.2), force_points=(0.5, 0.5))
        except:
            pass  # If adjust_text fails, labels will overlap but still visible
    
    # Regression line
    X = data['internet_usage'].values.reshape(-1, 1)
    y = data[mae_col].values
    
    lr = LinearRegression()
    lr.fit(X, y)
    
    x_line = np.linspace(data['internet_usage'].min(), data['internet_usage'].max(), 100)
    y_line = lr.predict(x_line.reshape(-1, 1))
    
    ax.plot(x_line, y_line, 'r--', linewidth=2, label='Linear fit')
    
    # Get stats
    r2 = aggregate_df[aggregate_df['Model'] == model.upper()]['R²'].values[0]
    pearson_r = aggregate_df[aggregate_df['Model'] == model.upper()]['Pearson r'].values[0]
    pearson_p = aggregate_df[aggregate_df['Model'] == model.upper()]['Pearson p'].values[0]
    slope = aggregate_df[aggregate_df['Model'] == model.upper()]['Slope'].values[0]
    
    # Add stats text
    sig = ""
    if pearson_p < 0.001:
        sig = "***"
    elif pearson_p < 0.01:
        sig = "**"
    elif pearson_p < 0.05:
        sig = "*"
    
    stats_text = f'r = {pearson_r:.3f}{sig}\nR² = {r2:.3f}\nSlope = {slope:.4f}'
    ax.text(0.05, 0.95, stats_text, transform=ax.transAxes, 
            verticalalignment='top', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    ax.set_xlabel('Internet Usage (%)', fontweight='bold', fontsize=11)
    ax.set_ylabel('Mean Absolute Error (pp)', fontweight='bold', fontsize=11)
    ax.set_title(f'{model.upper()}', fontweight='bold', fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('internet_usage_aggregate_regression.png', dpi=300, bbox_inches='tight')
plt.savefig('internet_usage_aggregate_regression.pdf', dpi=300, bbox_inches='tight')
print("\n✓ Saved internet_usage_aggregate_regression.png")
print("✓ Saved internet_usage_aggregate_regression.pdf")
plt.close()

# Figure 1b: Aggregate without labels (cleaner version)
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle('Internet Usage vs Prediction Error (Aggregated - No Labels)', 
             fontsize=16, fontweight='bold')

for idx, model in enumerate(models):
    ax = axes[idx // 2, idx % 2]
    
    mae_col = f'mae_{model}'
    data = country_aggregate[[mae_col, 'internet_usage']].dropna()
    
    # Scatter plot
    ax.scatter(data['internet_usage'], data[mae_col], 
              alpha=0.6, s=60, c='steelblue', edgecolors='black', linewidth=0.5)
    
    # Regression line
    X = data['internet_usage'].values.reshape(-1, 1)
    y = data[mae_col].values
    
    lr = LinearRegression()
    lr.fit(X, y)
    
    x_line = np.linspace(data['internet_usage'].min(), data['internet_usage'].max(), 100)
    y_line = lr.predict(x_line.reshape(-1, 1))
    
    ax.plot(x_line, y_line, 'r--', linewidth=2, label='Linear fit')
    
    # Get stats
    r2 = aggregate_df[aggregate_df['Model'] == model.upper()]['R²'].values[0]
    pearson_r = aggregate_df[aggregate_df['Model'] == model.upper()]['Pearson r'].values[0]
    pearson_p = aggregate_df[aggregate_df['Model'] == model.upper()]['Pearson p'].values[0]
    slope = aggregate_df[aggregate_df['Model'] == model.upper()]['Slope'].values[0]
    
    # Add stats text
    sig = ""
    if pearson_p < 0.001:
        sig = "***"
    elif pearson_p < 0.01:
        sig = "**"
    elif pearson_p < 0.05:
        sig = "*"
    
    stats_text = f'r = {pearson_r:.3f}{sig}\nR² = {r2:.3f}\nSlope = {slope:.4f}'
    ax.text(0.05, 0.95, stats_text, transform=ax.transAxes, 
            verticalalignment='top', fontsize=11,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    ax.set_xlabel('Internet Usage (%)', fontweight='bold', fontsize=11)
    ax.set_ylabel('Mean Absolute Error (pp)', fontweight='bold', fontsize=11)
    ax.set_title(f'{model.upper()}', fontweight='bold', fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('internet_usage_aggregate_regression_clean.png', dpi=300, bbox_inches='tight')
plt.savefig('internet_usage_aggregate_regression_clean.pdf', dpi=300, bbox_inches='tight')
print("✓ Saved internet_usage_aggregate_regression_clean.png")
print("✓ Saved internet_usage_aggregate_regression_clean.pdf")
plt.close()

# Figure 2: By-stage heatmap (Pearson r)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Internet Usage vs MAE: Stage-by-Stage Analysis', 
             fontsize=16, fontweight='bold')

# Panel A: Pearson r heatmap
ax = axes[0]
pivot_r = stage_df.pivot(index='Model', columns='Stage', values='Pearson r')
sns.heatmap(pivot_r, annot=True, fmt='.3f', cmap='RdBu_r', center=0, 
            cbar_kws={'label': 'Pearson r'}, ax=ax, vmin=-1, vmax=1)
ax.set_title('(a) Pearson Correlation by Stage', fontweight='bold', fontsize=12)
ax.set_xlabel('Stage', fontweight='bold', fontsize=11)
ax.set_ylabel('Model', fontweight='bold', fontsize=11)

# Panel B: Slope heatmap
ax = axes[1]
pivot_slope = stage_df.pivot(index='Model', columns='Stage', values='Slope')
sns.heatmap(pivot_slope, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            cbar_kws={'label': 'Slope (MAE change per 1% internet)'}, ax=ax)
ax.set_title('(b) Regression Slope by Stage', fontweight='bold', fontsize=12)
ax.set_xlabel('Stage', fontweight='bold', fontsize=11)
ax.set_ylabel('Model', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig('internet_usage_by_stage_heatmap.png', dpi=300, bbox_inches='tight')
plt.savefig('internet_usage_by_stage_heatmap.pdf', dpi=300, bbox_inches='tight')
print("✓ Saved internet_usage_by_stage_heatmap.png")
print("✓ Saved internet_usage_by_stage_heatmap.pdf")
plt.close()

# Figure 3: Stage-specific scatter plots with labels (for each model)
for model in models:
    fig, axes = plt.subplots(2, 4, figsize=(18, 10))
    fig.suptitle(f'{model.upper()}: Internet Usage vs MAE by Stage (with Country Labels)', 
                 fontsize=16, fontweight='bold')
    
    for stage_idx, stage in enumerate(sorted(df_complete['stage'].unique())):
        ax = axes[stage_idx // 4, stage_idx % 4]
        
        # Get data for this stage
        stage_data = df_complete[df_complete['stage'] == stage].copy()
        stage_country = stage_data.groupby('countrynew').agg({
            f'mae_{model}': 'mean',
            'internet_usage': 'first'
        }).reset_index()
        
        # Add country codes
        stage_country['country_code'] = stage_country['countrynew'].map(country_codes)
        
        data = stage_country[[f'mae_{model}', 'internet_usage', 'country_code']].dropna()
        
        if len(data) == 0:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', 
                   transform=ax.transAxes)
            ax.set_title(f'Stage {stage}')
            continue
        
        # Scatter plot
        ax.scatter(data['internet_usage'], data[f'mae_{model}'], 
                  alpha=0.6, s=30, c='steelblue', edgecolors='black', linewidth=0.5)
        
        # Add country labels (sample for readability)
        # Label countries with extreme values or every 5th country
        for idx, row in data.iterrows():
            if idx % 5 == 0 or row[f'mae_{model}'] > data[f'mae_{model}'].quantile(0.9) or \
               row[f'mae_{model}'] < data[f'mae_{model}'].quantile(0.1):
                ax.text(row['internet_usage'], row[f'mae_{model}'], 
                       row['country_code'], fontsize=5, alpha=0.6)
        
        # Regression line
        X = data['internet_usage'].values.reshape(-1, 1)
        y = data[f'mae_{model}'].values
        
        lr = LinearRegression()
        lr.fit(X, y)
        
        x_line = np.linspace(data['internet_usage'].min(), 
                            data['internet_usage'].max(), 100)
        y_line = lr.predict(x_line.reshape(-1, 1))
        
        ax.plot(x_line, y_line, 'r--', linewidth=1.5)
        
        # Get stats
        stage_stats = stage_df[(stage_df['Stage'] == stage) & 
                               (stage_df['Model'] == model.upper())]
        if len(stage_stats) > 0:
            r = stage_stats['Pearson r'].values[0]
            p = stage_stats['Pearson p'].values[0]
            slope = stage_stats['Slope'].values[0]
            
            sig = ""
            if p < 0.001:
                sig = "***"
            elif p < 0.01:
                sig = "**"
            elif p < 0.05:
                sig = "*"
            
            ax.text(0.05, 0.95, f'r={r:.2f}{sig}\nslope={slope:.3f}', 
                   transform=ax.transAxes, verticalalignment='top', fontsize=8,
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))
        
        ax.set_title(f'Stage {stage}', fontweight='bold', fontsize=10)
        ax.set_xlabel('Internet Usage (%)', fontsize=9)
        ax.set_ylabel('MAE (pp)', fontsize=9)
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'internet_usage_by_stage_{model}_labeled.png', dpi=300, bbox_inches='tight')
    plt.savefig(f'internet_usage_by_stage_{model}_labeled.pdf', dpi=300, bbox_inches='tight')
    print(f"✓ Saved internet_usage_by_stage_{model}_labeled.png")
    print(f"✓ Saved internet_usage_by_stage_{model}_labeled.pdf")
    plt.close()

# Figure 4: Stage-specific scatter plots WITHOUT labels (cleaner)
for model in models:
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    fig.suptitle(f'{model.upper()}: Internet Usage vs MAE by Stage', 
                 fontsize=16, fontweight='bold')
    
    for stage_idx, stage in enumerate(sorted(df_complete['stage'].unique())):
        ax = axes[stage_idx // 4, stage_idx % 4]
        
        # Get data for this stage
        stage_data = df_complete[df_complete['stage'] == stage].copy()
        stage_country = stage_data.groupby('countrynew').agg({
            f'mae_{model}': 'mean',
            'internet_usage': 'first'
        }).reset_index()
        
        data = stage_country[[f'mae_{model}', 'internet_usage']].dropna()
        
        if len(data) == 0:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', 
                   transform=ax.transAxes)
            ax.set_title(f'Stage {stage}')
            continue
        
        # Scatter plot
        ax.scatter(data['internet_usage'], data[f'mae_{model}'], 
                  alpha=0.6, s=35, c='steelblue', edgecolors='black', linewidth=0.5)
        
        # Regression line
        X = data['internet_usage'].values.reshape(-1, 1)
        y = data[f'mae_{model}'].values
        
        lr = LinearRegression()
        lr.fit(X, y)
        
        x_line = np.linspace(data['internet_usage'].min(), 
                            data['internet_usage'].max(), 100)
        y_line = lr.predict(x_line.reshape(-1, 1))
        
        ax.plot(x_line, y_line, 'r--', linewidth=2)
        
        # Get stats
        stage_stats = stage_df[(stage_df['Stage'] == stage) & 
                               (stage_df['Model'] == model.upper())]
        if len(stage_stats) > 0:
            r = stage_stats['Pearson r'].values[0]
            p = stage_stats['Pearson p'].values[0]
            slope = stage_stats['Slope'].values[0]
            
            sig = ""
            if p < 0.001:
                sig = "***"
            elif p < 0.01:
                sig = "**"
            elif p < 0.05:
                sig = "*"
            
            ax.text(0.05, 0.95, f'r={r:.2f}{sig}\nslope={slope:.3f}', 
                   transform=ax.transAxes, verticalalignment='top', fontsize=9,
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
        
        ax.set_title(f'Stage {stage}', fontweight='bold', fontsize=11)
        ax.set_xlabel('Internet Usage (%)', fontsize=9)
        ax.set_ylabel('MAE (pp)', fontsize=9)
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'internet_usage_by_stage_{model}_clean.png', dpi=300, bbox_inches='tight')
    plt.savefig(f'internet_usage_by_stage_{model}_clean.pdf', dpi=300, bbox_inches='tight')
    print(f"✓ Saved internet_usage_by_stage_{model}_clean.png")
    print(f"✓ Saved internet_usage_by_stage_{model}_clean.pdf")
    plt.close()

# ================================================================
# 7. Export Results
# ================================================================

print("\n" + "="*80)
print("7. EXPORTING RESULTS")
print("="*80)

# Save aggregate results
aggregate_df.to_csv('internet_usage_aggregate_regression.csv', index=False)
print("✓ Saved internet_usage_aggregate_regression.csv")

# Save by-stage results
stage_df.to_csv('internet_usage_by_stage_regression.csv', index=False)
print("✓ Saved internet_usage_by_stage_regression.csv")

# ================================================================
# 8. Summary
# ================================================================

print("\n" + "="*80)
print("SUMMARY")
print("="*80)

print("\n1. AGGREGATE RESULTS (Across All Stages):")
print("-" * 60)
for _, row in aggregate_df.iterrows():
    sig = ""
    if row['Pearson p'] < 0.001:
        sig = "***"
    elif row['Pearson p'] < 0.01:
        sig = "**"
    elif row['Pearson p'] < 0.05:
        sig = "*"
    
    direction = "NEGATIVE" if row['Slope'] < 0 else "POSITIVE"
    
    print(f"\n{row['Model']}:")
    print(f"   {direction} relationship: slope = {row['Slope']:.4f}")
    print(f"   Correlation: r = {row['Pearson r']:.3f} (p = {row['Pearson p']:.4f}) {sig}")
    print(f"   Variance explained: R² = {row['R²']:.3f}")
    
    if row['Slope'] < 0:
        print(f"   → Higher internet usage = LOWER prediction error")
    else:
        print(f"   → Higher internet usage = HIGHER prediction error")

print("\n\n2. BY-STAGE PATTERNS:")
print("-" * 60)

for model in models:
    model_stages = stage_df[stage_df['Model'] == model.upper()].sort_values('Stage')
    
    negative_stages = model_stages[model_stages['Slope'] < 0]['Stage'].tolist()
    positive_stages = model_stages[model_stages['Slope'] > 0]['Stage'].tolist()
    sig_stages = model_stages[model_stages['Pearson p'] < 0.05]['Stage'].tolist()
    
    print(f"\n{model.upper()}:")
    if negative_stages:
        print(f"   Negative relationship (stages): {negative_stages}")
    if positive_stages:
        print(f"   Positive relationship (stages): {positive_stages}")
    if sig_stages:
        print(f"   Significant at p<.05 (stages): {sig_stages}")
    else:
        print(f"   No significant relationships")

print("\n" + "="*80)
print("ANALYSIS COMPLETE!")
print("="*80)

print("\nFiles created:")
print("  - internet_usage_aggregate_regression.csv")
print("  - internet_usage_by_stage_regression.csv")
print("  - internet_usage_aggregate_regression.png / .pdf (with country labels)")
print("  - internet_usage_aggregate_regression_clean.png / .pdf (without labels)")
print("  - internet_usage_by_stage_heatmap.png / .pdf")
print("  - internet_usage_by_stage_gpt_labeled.png / .pdf")
print("  - internet_usage_by_stage_gpt_clean.png / .pdf")
print("  - internet_usage_by_stage_claude_labeled.png / .pdf")
print("  - internet_usage_by_stage_claude_clean.png / .pdf")
print("  - internet_usage_by_stage_gemini_labeled.png / .pdf")
print("  - internet_usage_by_stage_gemini_clean.png / .pdf")
print("  - internet_usage_by_stage_llama_labeled.png / .pdf")
print("  - internet_usage_by_stage_llama_clean.png / .pdf")
print("\nTotal: 2 CSV files + 26 image files (13 PNG + 13 PDF)")

INTERNET USAGE vs PREDICTION ERROR ANALYSIS

1. LOADING DATA
✓ Loaded predictions: 1000 rows
✓ Merged ground truth
✓ Loaded internet usage: 124 countries

⚠️  Warning: 4 countries missing internet usage:
   ['India', 'Kenya', 'Kosovo', 'Venezuela']

✓ Final dataset: 960 rows
   Countries: 120
   Internet usage range: 10.0% - 100.0%

2. CALCULATING MAE
✓ Calculated MAE for all models

3. AGGREGATE REGRESSION (Across All Stages)

✓ Aggregated data: 120 countries

GPT:
   N countries: 120
   Slope: -0.0742 (MAE change per 1% internet usage)
   Intercept: 15.78
   R²: 0.1531
   Pearson r: -0.3913, p = 0.0000
   Spearman ρ: -0.4315, p = 0.0000

CLAUDE:
   N countries: 120
   Slope: -0.0646 (MAE change per 1% internet usage)
   Intercept: 12.60
   R²: 0.0785
   Pearson r: -0.2802, p = 0.0019
   Spearman ρ: -0.5098, p = 0.0000

GEMINI:
   N countries: 120
   Slope: -0.1224 (MAE change per 1% internet usage)
   Intercept: 23.88
   R²: 0.2398
   Pearson r: -0.4897, p = 0.0000
   Spearman ρ: -0.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=feb9f195-de2a-416f-b8f1-09efca4e954f' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>